# AI-Powered Rice Leaf Disease Detection System

This notebook demonstrates the end-to-step process of loading image data, preprocessing, model building, and evaluation for rice leaf disease detection.

## Data Cleaning & Preparation

### Code from `src/data_loader.py`

In [ ]:
import os
import tensorflow as tf
import keras
from keras import layers, Sequential

def get_augmentation_layer():
    """Returns a sequential layer for image augmentation."""
    return Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
    ], name="data_augmentation")

def load_datasets(train_path, val_path, test_path, img_size=224, batch_size=8):
    """Loads and optimizes training, validation, and test datasets."""
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        train_path, image_size=(img_size, img_size), batch_size=batch_size, label_mode='categorical'
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        val_path, image_size=(img_size, img_size), batch_size=batch_size, label_mode='categorical'
    )
    test_ds = tf.keras.preprocessing.image_dataset_from_directory(
        test_path, image_size=(img_size, img_size), batch_size=batch_size, label_mode='categorical', shuffle=False
    )

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds, test_ds


## Visualization & EDA

### Code from `src/interpretability.py`

In [ ]:
import numpy as np
import keras
import keras.ops as ops

def find_layer_recursive(layer, name):
    """Recursively search for a layer by name in a model or layer."""
    if hasattr(layer, "name") and layer.name == name:
        return layer
    if hasattr(layer, "layers"):
        for sub_layer in layer.layers:
            found = find_layer_recursive(sub_layer, name)
            if found:
                return found
    return None

def find_last_conv_recursive(layer):
    """Recursively find the last Conv2D layer in a model or layer."""
    last_conv = None
    if isinstance(layer, keras.layers.Conv2D):
        last_conv = layer
    if hasattr(layer, "layers"):
        for sub_layer in layer.layers:
            found = find_last_conv_recursive(sub_layer)
            if found:
                last_conv = found
    return last_conv

def make_gradcam_heatmap(img_array, model, last_conv_layer_name="top_conv"):
    """Generates Grad-CAM heatmaps for model explainability."""
    
    # 1. Truly recursive layer lookup to find the target layer
    target_layer = find_layer_recursive(model, last_conv_layer_name)

    if not target_layer:
        # Fallback: Find the last Conv2D layer automatically
        target_layer = find_last_conv_recursive(model)

    if not target_layer:
        raise ValueError(f"Could not find any Conv2D layer in the model.")

    # 2. Compute Gradients based on backend
    backend = keras.backend.backend()
    
    if backend == "tensorflow":
        import tensorflow as tf
        # For TF, we still use the Functional Model approach as it's more stable there
        # but we ensure the model is functionalized if needed
        if not (hasattr(model, "input") and hasattr(model, "output")):
            inputs = keras.layers.Input(shape=img_array.shape[1:])
            outputs = model(inputs)
            func_model = keras.models.Model(inputs, outputs)
            target_layer = find_layer_recursive(func_model, last_conv_layer_name) or find_last_conv_recursive(func_model)
        else:
            func_model = model

        grad_model = keras.models.Model(func_model.inputs, [target_layer.output, func_model.output])
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_array)
            class_idx = ops.argmax(predictions[0])
            class_channel = predictions[:, class_idx]
        grads = tape.gradient(class_channel, conv_outputs)
        
    elif backend == "torch":
        import torch
        # For Torch, we use Forward Hooks to bypass Functional API connectivity issues
        activations = []
        def hook_fn(module, input, output):
            activations.append(output)
        
        # Attach hook to the target layer
        handle = target_layer.register_forward_hook(hook_fn)
        
        try:
            with torch.enable_grad():
                # Standardize input to numpy first to handle cross-framework tensors (TF -> Torch)
                img_numpy = ops.convert_to_numpy(img_array)
                img_tensor = torch.from_numpy(img_numpy).float()
                
                img_tensor.requires_grad = True # CRITICAL: Enable tracking for this image
                
                predictions = model(img_tensor)
                class_idx = ops.argmax(predictions[0])
                class_channel = predictions[:, class_idx]
                
                # activations[0] contains the output of target_layer
                conv_outputs = activations[0]
                grads = torch.autograd.grad(class_channel, conv_outputs)[0]
        finally:
            handle.remove() # Cleanup hook
        
    else:
        raise NotImplementedError(f"Grad-CAM gradient calculation not implemented for backend: {backend}")
    
    # 4. Generate Heatmap using Keras Ops (Backend-agnostic)
    pooled_grads = ops.mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., None]
    heatmap = ops.squeeze(heatmap)
    
    # ReLU and Normalization
    heatmap = ops.maximum(heatmap, 0.0)
    max_val = ops.max(heatmap)
    if max_val != 0:
        heatmap /= max_val
    
    return ops.convert_to_numpy(heatmap)


## Model Performance & Results

### Code from `src/model_builder.py`

In [ ]:
import keras
from keras import layers, Sequential
from keras.applications import MobileNetV2, EfficientNetB0

def build_transfer_model(base_model_class, num_classes, img_shape=(224, 224, 3), augmentation_layer=None):
    """Constructs a transfer learning model with a specified backbone."""
    base_model = base_model_class(weights='imagenet', include_top=False, input_shape=img_shape)
    base_model.trainable = False

    layers_list = [layers.Input(shape=img_shape)]
    if augmentation_layer:
        layers_list.append(augmentation_layer)
    
    layers_list.extend([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])

    return Sequential(layers_list)


### Code from `src/evaluation.py`

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def plot_confusion_matrix(y_true, y_pred, classes, save_path=None):
    """Plots and saves the confusion matrix heatmap."""
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title("Confusion Matrix: Rice Leaf Disease Prediction")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight')
        print(f"✅ Visualization saved to: {save_path}")
    plt.show()

def plot_training_history(history, save_path=None):
    """Plots and saves the accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='train')
    ax1.plot(history.history['val_accuracy'], label='val')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    
    # Loss
    ax2.plot(history.history['loss'], label='train')
    ax2.plot(history.history['val_loss'], label='val')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight')
        print(f"✅ Visualization saved to: {save_path}")
    plt.show()
